# LLM-Based Android App Permission Classification Pipeline

This notebook implements a modular pipeline for binary classification of Android app permissions using different Large Language Models (LLMs).

## Models to Test:
1. Llama 2
2. Mistral
3. FLAN-T5
4. Phi-3

## Target Permissions:
The pipeline will classify whether an app requests each of the specified permissions based on the app description.


## 1. Setup and Imports


In [ ]:
# Install required packages
%pip install transformers torch pandas numpy scikit-learn matplotlib seaborn accelerate bitsandbytes requests


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    pipeline, BitsAndBytesConfig
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
from typing import Dict, List, Tuple, Any
import time
from tqdm import tqdm
import requests

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


Using device: cuda


## 2. Configuration and Constants


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Configuration
CONFIG = {
    'data_path': '/content/drive/MyDrive/data/apps_with_permissions_cleaned_with_names_and_NEW_permissions_TOP30.csv',
    'target_permissions': [
        'Other : full network access',
        'Other : view network connections',
        'Other : prevent device from sleeping',
        'Photos/Media/Files : read the contents of your USB storage',
        'Storage : read the contents of your USB storage',
        'Uncategorized : receive data from Internet',
        'Photos/Media/Files : modify or delete the contents of your USB storage',
        'Storage : modify or delete the contents of your USB storage',
        'Other : control vibration',
        'Other : run at startup',
        'Camera : take pictures and videos',
        'Wi-Fi connection information : view Wi-Fi connections',
        'Location : precise location (GPS and network-based)',
        'Location : approximate location (network-based)',
        'Other : pair with Bluetooth devices',
        'Microphone : record audio',
        'Other : change your audio settings',
        'Other : access Bluetooth settings',
        'Calendar : read calendar events plus confidential information',
        "Calendar : add or modify calendar events and send email to guests without owners' knowledge",
        'Other : Google Play license check',
        'Contacts : read your contacts',
        'Device ID & call information : read phone status and identity',
        'Phone : read phone status and identity',
        'Other : draw over other apps',
        'Other : read Google service configuration',
        'Phone : directly call phone numbers',
        'Contacts : find accounts on the device',
        'Identity : find accounts on the device',
        'Other : connect and disconnect from Wi-Fi'
    ],
    'llm_models': {
        'llama2': 'meta-llama/Llama-2-7b-chat-hf', # 'TheBloke/Llama-2-7B-Chat-GGUF'
        'mistral': 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit', # 'mistralai/Mistral-7B-Instruct-v0.3'
        'flan-t5': 'google/flan-t5-large',
        'phi-3': 'leliuga/Phi-3-mini-4k-instruct-bnb-4bit', # 'microsoft/Phi-3-mini-4k-instruct'
        'llama3': 'unsloth/Meta-Llama-3.1-8B-bnb-4bit', # 'meta-llama/Llama-3-7b-hf'
    },
    'max_length': 512,
    'batch_size': 8,
    'test_size': 0.2,
    'random_state': 42
}

print(f"Target permissions: {len(CONFIG['target_permissions'])}")
print(f"LLM models: {list(CONFIG['llm_models'].keys())}")


Target permissions: 30
LLM models: ['llama2', 'mistral', 'flan-t5', 'phi-3', 'llama3']


## 3. Data Preprocessing Pipeline


In [ ]:
class DataPreprocessor:
    """
    Handles data loading, cleaning, and preprocessing for the permission classification task.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.data = None
        self.processed_data = None

    def load_data(self) -> pd.DataFrame:
        """
        Load the CSV data and perform initial cleaning.
        """
        print("Loading data...")
        self.data = pd.read_csv(self.config['data_path'])

        print(f"Data shape: {self.data.shape}")
        print(f"Columns: {len(self.data.columns)}")

        # Check if required columns exist
        required_cols = ['description'] + self.config['target_permissions']
        missing_cols = [col for col in required_cols if col not in self.data.columns]

        if missing_cols:
            print(f"Warning: Missing columns: {missing_cols}")
            # Filter to only include available permissions
            available_permissions = [col for col in self.config['target_permissions'] if col in self.data.columns]
            self.config['target_permissions'] = available_permissions
            print(f"Using available permissions: {len(available_permissions)}")

        return self.data

    def clean_text(self, text: str) -> str:
        """
        Clean and preprocess text data.
        """
        if pd.isna(text) or text == '':
            return ''

        # Remove HTML tags
        import re
        text = re.sub(r'<[^>]+>', '', str(text))

        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)

        # Remove special characters but keep basic punctuation
        text = re.sub(r'[^\w\s.,!?()-]', '', text)

        return text.strip()

    def preprocess_data(self) -> pd.DataFrame:
        """
        Preprocess the data for classification.
        """
        if self.data is None:
            self.load_data()

        print("Preprocessing data...")

        # Clean descriptions
        self.data['description'] = self.data['description'].apply(self.clean_text)

        # Remove rows with empty descriptions
        initial_count = len(self.data)
        self.data = self.data[self.data['description'].str.len() > 10]
        print(f"Removed {initial_count - len(self.data)} rows with empty/short descriptions")

        # Convert permission columns to binary (0/1)
        for perm in self.config['target_permissions']:
            if perm in self.data.columns:
                self.data[perm] = self.data[perm].astype(float).fillna(0)
                self.data[perm] = (self.data[perm] > 0).astype(int)

        self.processed_data = self.data.copy()

        print(f"Final data shape: {self.processed_data.shape}")
        print(f"Description length stats:")
        print(f"  Mean: {self.processed_data['description'].str.len().mean():.1f}")
        print(f"  Median: {self.processed_data['description'].str.len().median():.1f}")
        print(f"  Max: {self.processed_data['description'].str.len().max()}")

        return self.processed_data

    def get_class_distribution(self) -> Dict[str, Dict[str, int]]:
        """
        Get class distribution for each permission.
        """
        if self.processed_data is None:
            self.preprocess_data()

        distributions = {}
        for perm in self.config['target_permissions']:
            if perm in self.processed_data.columns:
                counts = self.processed_data[perm].value_counts()
                distributions[perm] = {
                    'class_0': counts.get(0, 0),
                    'class_1': counts.get(1, 0),
                    'total': len(self.processed_data)
                }

        return distributions

    def split_data(self, test_size: float = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Split data into train and test sets.
        """
        if self.processed_data is None:
            self.preprocess_data()

        test_size = test_size or self.config['test_size']

        train_data, test_data = train_test_split(
            self.processed_data,
            test_size=test_size,
            random_state=self.config['random_state'],
            stratify=None  # We'll handle stratification per permission later
        )

        print(f"Train set: {len(train_data)} samples")
        print(f"Test set: {len(test_data)} samples")

        return train_data, test_data

# Initialize preprocessor
preprocessor = DataPreprocessor(CONFIG)
data = preprocessor.load_data()
processed_data = preprocessor.preprocess_data()

# Display class distributions
distributions = preprocessor.get_class_distribution()
print("\nClass distributions:")
for perm, dist in list(distributions.items()):
    print(f"{perm[:50]}...: {dist['class_0']} (0) vs {dist['class_1']} (1)")


Loading data...
Data shape: (5348, 36)
Columns: 36
Preprocessing data...
Removed 0 rows with empty/short descriptions
Final data shape: (5348, 36)
Description length stats:
  Mean: 940.7
  Median: 626.0
  Max: 3456

Class distributions:
Other : full network access...: 40 (0) vs 5308 (1)
Other : view network connections...: 102 (0) vs 5246 (1)
Other : prevent device from sleeping...: 291 (0) vs 5057 (1)
Photos/Media/Files : read the contents of your USB...: 1026 (0) vs 4322 (1)
Storage : read the contents of your USB storage...: 1026 (0) vs 4322 (1)
Uncategorized : receive data from Internet...: 1041 (0) vs 4307 (1)
Photos/Media/Files : modify or delete the contents...: 1267 (0) vs 4081 (1)
Storage : modify or delete the contents of your US...: 1267 (0) vs 4081 (1)
Other : control vibration...: 1345 (0) vs 4003 (1)
Other : run at startup...: 1363 (0) vs 3985 (1)
Camera : take pictures and videos...: 1818 (0) vs 3530 (1)
Wi-Fi connection information : view Wi-Fi connecti...: 1915 (0) vs 

## 4. LLM Model Classes


In [ ]:
class LLMPermissionClassifier:
    """
    Base class for LLM-based permission classification.
    """

    def __init__(self, model_name: str, config: Dict[str, Any]):
        self.model_name = model_name
        self.config = config
        self.model = None
        self.tokenizer = None
        self.pipeline = None
        self.device = device

    def load_model(self):
        """
        Load the model and tokenizer. To be implemented by subclasses.
        """
        raise NotImplementedError

    def create_prompt(self, description: str, permission: str) -> str:
        """
        Create a prompt for the classification task.
        """
        prompt = f"""
Task: Determine if an Android app requests the following permission based on its description.

App Description: {description}

Permission: {permission}

Question: Does this app request the permission "{permission}"?

Answer with only "YES" or "NO":"""
        return prompt.strip()

    def classify_permission(self, description: str, permission: str) -> tuple:
        """
        Classify whether an app requests a specific permission.
        Returns (prediction, raw_response) where prediction is 1 for YES, 0 for NO.
        """
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, permission)

        try:
            response = self.pipeline(prompt, max_length=50, do_sample=False)
            if isinstance(response, list):
                response_text = response[0]['generated_text']
            else:
                response_text = response['generated_text']

            # Store raw response before processing
            raw_response = response_text.strip()


            # Extract the answer from the response
            response_text = response_text.lower().strip()

            print(f'Response: {response_text}')
            if 'yes' in response_text:
                return (1, raw_response)
            elif 'no' in response_text:
                return (0, raw_response)
            else:
                # Default to 0 if unclear
                return (0, raw_response)

        except Exception as e:
            print(f"Error in classification: {e}")
            return (0, f"Error: {str(e)}")

    def batch_classify(self, descriptions: List[str], permission: str) -> tuple:
        """
        Classify multiple descriptions for a single permission.
        Returns (predictions, raw_responses) where predictions is List[int] and raw_responses is List[str].
        """
        predictions = []
        raw_responses = []

        for desc in tqdm(descriptions, desc=f"Classifying {permission[:30]}..."):
            pred, raw_resp = self.classify_permission(desc, permission)
            predictions.append(pred)
            raw_responses.append(raw_resp)

        return predictions, raw_responses

class Llama2Classifier(LLMPermissionClassifier):

    def load_model(self):
        print(f"Loading {self.model_name} (Llama 2 via HF + bnb 4-bit)...")

        # 4-bit NF4 quantization config
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        import os
        hf_token = os.environ.get('HF_TOKEN')
        model_id = self.config['llm_models']['llama2']

        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=hf_token,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token is None and hasattr(self.tokenizer, 'eos_token'):
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Quantized model
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=hf_token,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        # Simple pipeline; decoding is controlled in classify_permission
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def classify_permission(self, description: str, permission: str) -> tuple:
        """
        Use max_new_tokens + truncation to avoid max_length issues.
        Deterministic decoding for classification.
        Returns (prediction, raw_response) where prediction is 1 for YES, 0 for NO.
        """
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, permission)

        try:
            response = self.pipeline(
                prompt,
                max_new_tokens=10,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )

            if isinstance(response, list):
                response_text = response[0].get('generated_text', '')
            else:
                response_text = response.get('generated_text', '')

            # Store raw response before processing
            raw_response = response_text.strip()

            response_text = response_text.lower().strip()

            if 'yes' in response_text:
                return (1, raw_response)
            if 'no' in response_text:
                return (0, raw_response)
            return (0, raw_response)

        except Exception as e:
            print(f"Error in classification: {e}")
            return (0, f"Error: {str(e)}")

class MistralClassifier(LLMPermissionClassifier):
    """
    Mistral based classifier.
    """

    def load_model(self):
        """
        Load Mistral model.
        """
        print(f"Loading {self.model_name}...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config['llm_models']['mistral'],
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.config['llm_models']['mistral'],
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def create_prompt(self, description: str, permission: str) -> str:
        """
        Override create_prompt to use Mistral's chat template format.
        """
        prompt_text = f"""Task: Determine if an Android app requests the following permission based on its description.

    App Description: {description}

    Permission: {permission}

    Question: Does this app request the permission "{permission}"?

    Answer with only "YES" or "NO":"""

        # Apply Mistral chat template if available
        if hasattr(self.tokenizer, 'apply_chat_template') and hasattr(self.tokenizer, 'chat_template') and self.tokenizer.chat_template:
            messages = [{"role": "user", "content": prompt_text}]
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            # Fallback format for Mistral
            prompt = f"[INST] {prompt_text} [/INST]"

        return prompt

    def classify_permission(self, description: str, permission: str) -> tuple:
        """
        Override classify_permission to use max_new_tokens and better response parsing.
        Returns (prediction, raw_response) where prediction is 1 for YES, 0 for NO.
        """
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, permission)

        try:
            # Generate with max_new_tokens (not max_length) and deterministic settings
            response = self.pipeline(
                prompt,
                max_new_tokens=10,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False  # Only return newly generated text
            )

            # Extract response text
            if isinstance(response, list):
                response_text = response[0].get('generated_text', '')
            else:
                response_text = response.get('generated_text', '')

            # Store raw response before processing
            raw_response = response_text.strip()

            # Clean and normalize response - only check first few words
            response_text = response_text.lower().strip()

            # Split and only check first 5 words to avoid prompt contamination
            words = response_text.split()[:5]
            response_text = ' '.join(words)

            # Check for YES/NO - prioritize NO if both present
            if 'no' in response_text:
                return (0, raw_response)
            elif 'yes' in response_text:
                return (1, raw_response)
            else:
                # Default to 0 if unclear
                return (0, raw_response)

        except Exception as e:
            print(f"Error in Mistral classification: {e}")
            return (0, f"Error: {str(e)}")


class FlanT5Classifier(LLMPermissionClassifier):
    """
    FLAN-T5 based classifier.
    """

    def load_model(self):
        """
        Load FLAN-T5 model.
        """
        print(f"Loading {self.model_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config['llm_models']['flan-t5']
        )

        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config['llm_models']['flan-t5'],
            device_map="auto"
        )

        self.pipeline = pipeline(
            "text2text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")


class Phi3Classifier(LLMPermissionClassifier):
    """
    Phi-3 based classifier.
    """

    def load_model(self):
        """
        Load Phi-3 model.
        """
        print(f"Loading {self.model_name}...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config['llm_models']['phi-3'],
            trust_remote_code=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.config['llm_models']['phi-3'],
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=False # set as False for phi-3
        )

        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

class Llama3Classifier(LLMPermissionClassifier):

    def load_model(self):
        print(f"Loading {self.model_name} (Llama 3 via HF + bnb 4-bit)...")

        # 4-bit NF4 quantization config
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        import os
        hf_token = os.environ.get('HF_TOKEN')
        model_id = self.config['llm_models']['llama3']

        # Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=hf_token,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token is None and hasattr(self.tokenizer, 'eos_token'):
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Quantized model
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=hf_token,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        # Simple pipeline; decoding is controlled in classify_permission
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto"
        )

        print(f"{self.model_name} loaded successfully!")

    def classify_permission(self, description: str, permission: str) -> tuple:
        """
        Use max_new_tokens + truncation to avoid max_length issues.
        Deterministic decoding for classification.
        Returns (prediction, raw_response) where prediction is 1 for YES, 0 for NO.
        """
        if self.pipeline is None:
            self.load_model()

        prompt = self.create_prompt(description, permission)

        try:
            response = self.pipeline(
                prompt,
                max_new_tokens=10,
                do_sample=False,
                temperature=0.0,
                truncation=True,
                return_full_text=False
            )

            if isinstance(response, list):
                response_text = response[0].get('generated_text', '')
            else:
                response_text = response.get('generated_text', '')

            # Store raw response before processing
            raw_response = response_text.strip()

            response_text = response_text.lower().strip()

            if 'yes' in response_text:
                return (1, raw_response)
            if 'no' in response_text:
                return (0, raw_response)
            return (0, raw_response)

        except Exception as e:
            print(f"Error in classification: {e}")
            return (0, f"Error: {str(e)}")

## 5. Evaluation Metrics and Main Pipeline


In [ ]:
class EvaluationMetrics:
    """
    Handles evaluation metrics and visualization.
    """

    @staticmethod
    def calculate_metrics(y_true: List[int], y_pred: List[int]) -> Dict[str, float]:
        """
        Calculate classification metrics.
        """
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0)
        }

        return metrics

    @staticmethod
    def plot_confusion_matrix(y_true: List[int], y_pred: List[int],
                            title: str = "Confusion Matrix"):
        """
        Plot confusion matrix.
        """
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['No Permission', 'Has Permission'],
                   yticklabels=['No Permission', 'Has Permission'])
        plt.title(title)
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()

    @staticmethod
    def plot_metrics_comparison(results: Dict[str, Dict[str, Dict[str, float]]]):
        """
        Plot comparison of metrics across models and permissions.
        """
        # Prepare data for plotting
        models = list(results.keys())
        permissions = list(results[models[0]].keys())
        metrics = ['accuracy', 'precision', 'recall', 'f1']

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()

        for i, metric in enumerate(metrics):
            ax = axes[i]

            # Create data matrix
            data_matrix = np.zeros((len(models), len(permissions)))

            for j, model in enumerate(models):
                for k, perm in enumerate(permissions):
                    data_matrix[j, k] = results[model][perm][metric]

            # Plot heatmap
            im = ax.imshow(data_matrix, cmap='viridis', aspect='auto')

            # Set labels
            ax.set_xticks(range(len(permissions)))
            ax.set_xticklabels([p[:20] + '...' if len(p) > 20 else p for p in permissions],
                              rotation=45, ha='right')
            ax.set_yticks(range(len(models)))
            ax.set_yticklabels(models)

            # Add colorbar
            plt.colorbar(im, ax=ax)

            # Add text annotations
            for j in range(len(models)):
                for k in range(len(permissions)):
                    text = ax.text(k, j, f'{data_matrix[j, k]:.3f}',
                                 ha="center", va="center", color="white")

            ax.set_title(f'{metric.capitalize()} Score')

        plt.tight_layout()
        plt.show()

    @staticmethod
    def create_summary_table(results: Dict[str, Dict[str, Dict[str, float]]]) -> pd.DataFrame:
        """
        Create a summary table of results.
        """
        summary_data = []

        for model, model_results in results.items():
            for permission, metrics in model_results.items():
                summary_data.append({
                    'Model': model,
                    'Permission': permission,
                    'Accuracy': metrics['accuracy'],
                    'Precision': metrics['precision'],
                    'Recall': metrics['recall'],
                    'F1-Score': metrics['f1']
                })

        return pd.DataFrame(summary_data)


class PermissionClassificationPipeline:
    """
    Main pipeline for running permission classification experiments.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.preprocessor = DataPreprocessor(config)
        self.evaluator = EvaluationMetrics()
        self.results = {}

    def run_experiment(self, model_name: str, sample_size: int = None) -> Dict[str, Dict[str, float]]:
        """
        Run classification experiment for a specific model.

        Args:
            model_name: Name of the LLM model to use
            sample_size: Number of samples to use (None for all data)

        Returns:
            Dictionary with results for each permission
        """
        print(f"\n{'='*60}")
        print(f"Running experiment with {model_name}")
        print(f"{'='*60}")

        # Initialize classifier based on model name
        if model_name.lower() == 'llama2':
            classifier = Llama2Classifier(model_name, self.config)
        elif model_name.lower() == 'mistral':
            classifier = MistralClassifier(model_name, self.config)
        elif model_name.lower() == 'flan-t5':
            classifier = FlanT5Classifier(model_name, self.config)
        elif model_name.lower() == 'phi-3':
            classifier = Phi3Classifier(model_name, self.config)
        elif model_name.lower() == 'llama3':
            classifier = Llama3Classifier(model_name, self.config)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        # Load and preprocess data
        data = self.preprocessor.preprocess_data()

        # Sample data if specified
        if sample_size and sample_size < len(data):
            data = data.sample(n=sample_size, random_state=self.config['random_state'])
            print(f"Using sample of {sample_size} records")

        # Split data
        train_data, test_data = self.preprocessor.split_data()

        # Get test descriptions and labels
        test_descriptions = test_data['description'].tolist()

        # Run classification for each permission
        model_results = {}

        for permission in self.config['target_permissions']:
            if permission not in test_data.columns:
                print(f"Skipping {permission} - not in data")
                continue

            print(f"\nClassifying permission: {permission}")

            # Get true labels
            y_true = test_data[permission].tolist()

            # Get predictions (and raw responses, though not used here)
            y_pred, _ = classifier.batch_classify(test_descriptions, permission)

            # Calculate metrics
            metrics = self.evaluator.calculate_metrics(y_true, y_pred)
            model_results[permission] = metrics

            print(f"Results for {permission}:")
            print(f"  Accuracy: {metrics['accuracy']:.3f}")
            print(f"  Precision: {metrics['precision']:.3f}")
            print(f"  Recall: {metrics['recall']:.3f}")
            print(f"  F1-Score: {metrics['f1']:.3f}")

            # Show class distribution
            true_pos = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
            true_neg = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
            false_pos = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
            false_neg = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

            print(f"  Confusion Matrix: TP={true_pos}, TN={true_neg}, FP={false_pos}, FN={false_neg}")

        # Store results
        self.results[model_name] = model_results

        return model_results

    def run_all_experiments(self, sample_size: int = None):
        """
        Run experiments for all models.
        """
        for model_name in self.config['llm_models'].keys():
            try:
                self.run_experiment(model_name, sample_size)
            except Exception as e:
                print(f"Error running experiment for {model_name}: {e}")
                continue

    def visualize_results(self):
        """
        Visualize results from all experiments.
        """
        if not self.results:
            print("No results to visualize. Run experiments first.")
            return

        # Plot metrics comparison
        self.evaluator.plot_metrics_comparison(self.results)

        # Create summary table
        summary_df = self.evaluator.create_summary_table(self.results)

        print("\nSummary Results:")
        print(summary_df.to_string(index=False))

        return summary_df

    def save_results(self, filename: str = 'permission_classification_results.json'):
        """
        Save results to file.
        """
        with open(filename, 'w') as f:
            json.dump(self.results, f, indent=2)
        print(f"Results saved to {filename}")

# Initialize pipeline
classification_pipeline = PermissionClassificationPipeline(CONFIG)


### Usage Examples


In [ ]:
# Example 1: Run experiment for a single model
# Uncomment the line below to run Llama 3.1 experiment
# results_llama = classification_pipeline.run_experiment('llama3.1', sample_size=100)  # Use sample for testing

# Example 2: Run experiment for FLAN-T5 (smaller model, faster)
# results_flan = classification_pipeline.run_experiment('flan-t5', sample_size=100)

# Example 3: Run all experiments (this will take a long time)
# classification_pipeline.run_all_experiments(sample_size=100)

# Example 4: Visualize results
# summary_df = classification_pipeline.visualize_results()

# Example 5: Save results
# classification_pipeline.save_results('my_results.json')

print("Pipeline ready! Uncomment the examples above to run experiments.")
print("\nNote: Start with small sample sizes for testing, then scale up.")
print("Recommended order for testing:")
print("1. FLAN-T5 (fastest, good baseline)")
print("2. Phi-3 (smaller model)")
print("3. Mistral (medium size)")
print("4. Llama 3.1 (largest, most capable)")


Pipeline ready! Uncomment the examples above to run experiments.

Note: Start with small sample sizes for testing, then scale up.
Recommended order for testing:
1. FLAN-T5 (fastest, good baseline)
2. Phi-3 (smaller model)
3. Mistral (medium size)
4. Llama 3.1 (largest, most capable)


### Advanced Usage and Customization


In [ ]:
# Custom prompt template
def create_custom_prompt(description: str, permission: str) -> str:
    """
    Create a custom prompt for classification.
    """
    return f"""
You are an expert Android security analyst. Analyze the following app description and determine if the app requests the specified permission.

App Description: {description}

Permission to check: {permission}

Based on the app's functionality described above, does this app need the permission "{permission}"?

Consider:
- The app's primary purpose
- Features mentioned in the description
- Whether the permission is necessary for the described functionality

Answer: YES or NO"""

# Function to test a single permission with custom parameters
def test_single_permission(model_name: str, permission: str, sample_size: int = 50):
    """
    Test classification for a single permission.
    """
    print(f"Testing {model_name} on {permission}")

    # Get data
    data = preprocessor.preprocess_data()
    test_data = data.sample(n=min(sample_size, len(data)), random_state=42)

    # Initialize classifier
    if model_name.lower() == 'flan-t5':
        classifier = FlanT5Classifier(model_name, CONFIG)
    elif model_name.lower() == 'phi-3':
        classifier = Phi3Classifier(model_name, CONFIG)
    else:
        print(f"Model {model_name} not implemented in this example")
        return

    # Get predictions (and raw responses, though not used here)
    descriptions = test_data['description'].tolist()
    y_true = test_data[permission].tolist()
    y_pred, _ = classifier.batch_classify(descriptions, permission)

    # Calculate metrics
    metrics = EvaluationMetrics.calculate_metrics(y_true, y_pred)

    print(f"Results:")
    for metric, value in metrics.items():
        print(f"  {metric.capitalize()}: {value:.3f}")

    return metrics

# Memory management tips
def clear_memory():
    """
    Clear GPU memory.
    """
    import gc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("Memory cleared")

# Check available memory
def check_memory():
    """
    Check available GPU memory.
    """
    if torch.cuda.is_available():
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
        print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1e9:.1f} GB")
    else:
        print("CUDA not available")

# Example usage of custom functions
print("Custom functions defined. Examples:")
print("# Test FLAN-T5 on camera permission:")
print("# test_single_permission('flan-t5', 'Camera : take pictures and videos', 50)")
print("\n# Test Phi-3 on network access:")
print("# test_single_permission('phi-3', 'Other : full network access', 50)")
print("\n# Check memory usage:")
print("# check_memory()")
print("\n# Clear memory if needed:")
print("# clear_memory()")

# Check current memory status
check_memory()


Custom functions defined. Examples:
# Test FLAN-T5 on camera permission:
# test_single_permission('flan-t5', 'Camera : take pictures and videos', 50)

# Test Phi-3 on network access:
# test_single_permission('phi-3', 'Other : full network access', 50)

# Check memory usage:
# check_memory()

# Clear memory if needed:
# clear_memory()
GPU Memory: 15.8 GB
GPU Memory Allocated: 0.0 GB
GPU Memory Cached: 0.0 GB


## 6. Enhanced Pipeline with Google Drive Integration

In [ ]:
import os
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

class GoogleDrivePermissionClassificationPipeline(PermissionClassificationPipeline):
    """
    Enhanced pipeline that creates model-specific folders and saves results to Google Drive.
    """

    def __init__(self, config: Dict[str, Any]):
        super().__init__(config)
        self.base_path = "/content/drive/MyDrive/LLMs/binary_classification/results"
        self.setup_google_drive()

    def setup_google_drive(self):
        """
        Mount Google Drive and create model-specific directories.
        """
        try:
            # Mount Google Drive
            drive.mount('/content/drive', force_remount=True)

            # Create base directory structure
            os.makedirs(self.base_path, exist_ok=True)

            # Create model-specific directories for each model in config
            for model_name in self.config['llm_models'].keys():
                model_path = f"{self.base_path}/{model_name}"
                os.makedirs(f"{model_path}/json_results", exist_ok=True)
                os.makedirs(f"{model_path}/permission_plots", exist_ok=True)
                os.makedirs(f"{model_path}/visualizations", exist_ok=True)
                os.makedirs(f"{model_path}/predictions", exist_ok=True)
                print(f"✅ Created directories for {model_name}")

            print(f"✅ Google Drive mounted and model-specific directories created at: {self.base_path}")

        except Exception as e:
            print(f"⚠️  Google Drive setup failed: {e}")
            print("Using local storage instead...")
            self.base_path = "./results"
            os.makedirs(self.base_path, exist_ok=True)

    def get_model_path(self, model_name: str) -> str:
        """
        Get the model-specific path for saving results.
        """
        return f"{self.base_path}/{model_name}"

    def save_permission_results(self, model_name: str, permission: str, metrics: Dict[str, float],
                              y_true: List[int], y_pred: List[int]):
        """
        Save results and visualization for a single permission in model-specific folder.
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_path = self.get_model_path(model_name)

        # 1. Save JSON results
        if model_name not in self.results:
            self.results[model_name] = {}

        self.results[model_name][permission] = metrics

        # Save individual permission results
        permission_data = {
            "model": model_name,
            "permission": permission,
            "timestamp": timestamp,
            "metrics": metrics,
            "sample_size": len(y_true),
            "class_distribution": {
                "true_positive": sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1),
                "true_negative": sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0),
                "false_positive": sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1),
                "false_negative": sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
            }
        }

        # Save individual permission JSON in model folder
        safe_permission_name = permission.replace(":", "_").replace(" ", "_").replace("/", "_")
        json_file = f"{model_path}/json_results/{safe_permission_name}_{timestamp}.json"
        with open(json_file, 'w') as f:
            json.dump(permission_data, f, indent=2)

        # Save overall results for this model
        overall_json = f"{model_path}/json_results/{model_name}_overall_results.json"
        new_data = self.results[model_name]

        # If file exists, read existing data
        if os.path.exists(overall_json):
            with open(overall_json, 'r') as f:
                try:
                    existing_data = json.load(f)
                except json.JSONDecodeError:
                    existing_data = {}  # empty or corrupted file fallback
        else:
            existing_data = {}

        # Merge or append new data
        if isinstance(existing_data, dict) and isinstance(new_data, dict):
            existing_data.update(new_data)
        elif isinstance(existing_data, list):
            existing_data.append(new_data)
        else:
            existing_data = [existing_data, new_data]

        # Write back to file
        with open(overall_json, 'w') as f:
            json.dump(existing_data, f, indent=2)

        # 2. Create and save visualization in model folder
        self.create_permission_visualization(model_name, permission, metrics, y_true, y_pred, timestamp, model_path)

        print(f"  💾 Results saved: {json_file}")
        print(f"  📊 Visualization saved: {model_path}/permission_plots/{safe_permission_name}_{timestamp}.png")

    def create_permission_visualization(self, model_name: str, permission: str, metrics: Dict[str, float],
                                      y_true: List[int], y_pred: List[int], timestamp: str, model_path: str):
        """
        Create visualization for a single permission in model-specific folder.
        """
        safe_permission_name = permission.replace(":", "_").replace(" ", "_").replace("/", "_")

        # Create figure with subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'{model_name} - {permission}\nTimestamp: {timestamp}', fontsize=16, fontweight='bold')

        # 1. Confusion Matrix
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
                   xticklabels=['No Permission', 'Has Permission'],
                   yticklabels=['No Permission', 'Has Permission'])
        axes[0,0].set_title('Confusion Matrix')
        axes[0,0].set_ylabel('True Label')
        axes[0,0].set_xlabel('Predicted Label')

        # 2. Metrics Bar Chart
        metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
        metric_values = [metrics['accuracy'], metrics['precision'], metrics['recall'], metrics['f1']]
        bars = axes[0,1].bar(metric_names, metric_values, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
        axes[0,1].set_title('Performance Metrics')
        axes[0,1].set_ylabel('Score')
        axes[0,1].set_ylim(0, 1)

        # Add value labels on bars
        for bar, value in zip(bars, metric_values):
            axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        # 3. Class Distribution
        class_counts = [sum(1 for x in y_true if x == 0), sum(1 for x in y_true if x == 1)]
        axes[1,0].pie(class_counts, labels=['No Permission', 'Has Permission'],
                     autopct='%1.1f%%', colors=['lightblue', 'lightcoral'])
        axes[1,0].set_title('True Class Distribution')

        # 4. Prediction Distribution
        pred_counts = [sum(1 for x in y_pred if x == 0), sum(1 for x in y_pred if x == 1)]
        axes[1,1].pie(pred_counts, labels=['Predicted No', 'Predicted Yes'],
                     autopct='%1.1f%%', colors=['lightgreen', 'orange'])
        axes[1,1].set_title('Predicted Class Distribution')

        plt.tight_layout()

        # Save the plot in model-specific folder
        plot_file = f"{model_path}/permission_plots/{safe_permission_name}_{timestamp}.png"
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        plt.close()  # Close to free memory

    def run_experiment_with_google_drive_saving(self, model_name: str, sample_size: int = None, index: int = 0) -> Dict[str, Dict[str, float]]:
        """
        Run classification experiment with Google Drive saving after each permission.
        """
        print(f"\n{'='*60}")
        print(f"Running experiment with {model_name} (with Google Drive saving)")
        model_path = self.get_model_path(model_name)
        print(f"Results will be saved to: {model_path}")
        print(f"{'='*60}")

        # Initialize classifier based on model name
        if model_name.lower() == 'llama2':
            classifier = Llama2Classifier(model_name, self.config)
        elif model_name.lower() == 'mistral':
            classifier = MistralClassifier(model_name, self.config)
        elif model_name.lower() == 'flan-t5':
            classifier = FlanT5Classifier(model_name, self.config)
        elif model_name.lower() == 'phi-3':
            classifier = Phi3Classifier(model_name, self.config)
        elif model_name.lower() == 'llama3':
            classifier = Llama3Classifier(model_name, self.config)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        # Load and preprocess data
        data = self.preprocessor.preprocess_data()

        # Sample data if specified
        if sample_size and sample_size < len(data):
            data = data.sample(n=sample_size, random_state=self.config['random_state'])
            print(f"Using sample of {sample_size} records")

        # Split data
        train_data, test_data = self.preprocessor.split_data()

        # Get test descriptions and labels
        test_descriptions = test_data['description'].tolist()

        # Initialize model results
        model_results = {}

        # Process each permission with Google Drive saving
        for i, permission in enumerate(self.config['target_permissions'][index:]):
            if permission not in test_data.columns:
                print(f"Skipping {permission} - not in data")
                continue

            print(f"\n[{i+1}/{len(self.config['target_permissions'])}] Classifying permission: {permission}")

            try:
                # Get true labels
                y_true = test_data[permission].tolist()

                # Get predictions and raw responses
                y_pred, raw_responses = classifier.batch_classify(test_descriptions, permission)

                # Build predictions DataFrame for export
                try:
                    cols = []
                    for c in ['appId', 'description', 'name']:
                        if c in test_data.columns:
                            cols.append(c)
                    pred_df = test_data[cols].copy() if cols else test_data[['description']].copy()
                    pred_df['true_label'] = y_true
                    pred_df['prediction'] = y_pred
                    pred_df['llm_raw_response'] = raw_responses
                    # Save predictions Excel
                    safe_permission_name = permission.replace(":", "_").replace(" ", "_").replace("/", "_")
                    model_path = self.get_model_path(model_name)
                    from datetime import datetime
                    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                    pred_path = f"{model_path}/predictions/{safe_permission_name}_{ts}.xlsx"
                    pred_df.to_excel(pred_path, index=False)
                    print(f"  📄 Predictions saved: {pred_path}")
                except Exception as e:
                    print(f"  ⚠️ Could not save predictions Excel: {e}")

                # Calculate metrics
                metrics = self.evaluator.calculate_metrics(y_true, y_pred)
                model_results[permission] = metrics

                # Save results and visualization to Google Drive
                self.save_permission_results(model_name, permission, metrics, y_true, y_pred)

                print(f"✓ Completed {permission}: Accuracy={metrics['accuracy']:.3f}, F1={metrics['f1']:.3f}")

            except KeyboardInterrupt:
                print(f"\n⚠️  Interrupted during {permission}. Results saved up to this point.")
                print(f"💾 Check Google Drive folder: {model_path}")
                return model_results
            except Exception as e:
                print(f"❌ Error processing {permission}: {e}")
                continue

        # Create final summary visualization in model folder
        self.create_final_summary_visualization(model_name, model_results, model_path)

        print(f"\n✅ All permissions completed for {model_name}!")
        print(f"📁 All results saved to: {model_path}")
        return model_results

    def create_final_summary_visualization(self, model_name: str, model_results: Dict[str, Dict[str, float]], model_path: str):
        """
        Create final summary visualization for all permissions in model-specific folder.
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Prepare data for visualization
        permissions = list(model_results.keys())
        metrics = ['accuracy', 'precision', 'recall', 'f1']

        # Create comprehensive summary plot
        fig, axes = plt.subplots(2, 2, figsize=(20, 15))
        fig.suptitle(f'{model_name} - Complete Results Summary\nTimestamp: {timestamp}',
                    fontsize=16, fontweight='bold')

        # 1. Metrics heatmap
        data_matrix = np.zeros((len(permissions), len(metrics)))
        for i, perm in enumerate(permissions):
            for j, metric in enumerate(metrics):
                data_matrix[i, j] = model_results[perm][metric]

        im = axes[0,0].imshow(data_matrix, cmap='viridis', aspect='auto')
        axes[0,0].set_xticks(range(len(metrics)))
        axes[0,0].set_xticklabels([m.capitalize() for m in metrics])
        axes[0,0].set_yticks(range(len(permissions)))
        axes[0,0].set_yticklabels([p[:30] + '...' if len(p) > 30 else p for p in permissions], fontsize=8)
        axes[0,0].set_title('Performance Heatmap')

        # Add colorbar
        plt.colorbar(im, ax=axes[0,0])

        # 2. Average metrics
        avg_metrics = {metric: np.mean([model_results[perm][metric] for perm in permissions])
                      for metric in metrics}
        bars = axes[0,1].bar(avg_metrics.keys(), avg_metrics.values(),
                            color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
        axes[0,1].set_title('Average Performance Across All Permissions')
        axes[0,1].set_ylabel('Score')
        axes[0,1].set_ylim(0, 1)

        for bar, value in zip(bars, avg_metrics.values()):
            axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        # 3. Accuracy distribution
        accuracies = [model_results[perm]['accuracy'] for perm in permissions]
        axes[1,0].hist(accuracies, bins=10, alpha=0.7, color='skyblue', edgecolor='black')
        axes[1,0].set_title('Accuracy Distribution')
        axes[1,0].set_xlabel('Accuracy')
        axes[1,0].set_ylabel('Number of Permissions')
        axes[1,0].axvline(np.mean(accuracies), color='red', linestyle='--',
                         label=f'Mean: {np.mean(accuracies):.3f}')
        axes[1,0].legend()

        # 4. F1-Score vs Accuracy scatter
        f1_scores = [model_results[perm]['f1'] for perm in permissions]
        scatter = axes[1,1].scatter(accuracies, f1_scores, alpha=0.7, s=100, c=range(len(permissions)),
                                   cmap='viridis')
        axes[1,1].set_title('F1-Score vs Accuracy')
        axes[1,1].set_xlabel('Accuracy')
        axes[1,1].set_ylabel('F1-Score')
        axes[1,1].plot([0, 1], [0, 1], 'r--', alpha=0.5)  # Diagonal line

        plt.tight_layout()

        # Save the summary plot in model-specific folder
        summary_file = f"{model_path}/visualizations/{model_name}_summary_{timestamp}.png"
        plt.savefig(summary_file, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"📊 Final summary visualization saved: {summary_file}")

# Create Google Drive enabled pipeline
gdrive_pipeline = GoogleDrivePermissionClassificationPipeline(CONFIG)

print(f"📁 Base path: {gdrive_pipeline.base_path}")
print("Use gdrive_pipeline.run_experiment_with_google_drive_saving() for execution.")


Mounted at /content/drive
✅ Created directories for llama2
✅ Created directories for mistral
✅ Created directories for flan-t5
✅ Created directories for phi-3
✅ Created directories for llama3
✅ Google Drive mounted and model-specific directories created at: /content/drive/MyDrive/LLMs/binary_classification/results
📁 Base path: /content/drive/MyDrive/LLMs/binary_classification/results
Use gdrive_pipeline.run_experiment_with_google_drive_saving() for execution.


## 7. Run models

In [ ]:
# results = gdrive_pipeline.run_experiment_with_google_drive_saving('flan-t5', sample_size=100, index=0)

In [ ]:
# results = gdrive_pipeline.run_experiment_with_google_drive_saving('phi-3', sample_size=100, index=0)

In [ ]:
# results = gdrive_pipeline.run_experiment_with_google_drive_saving('llama2', sample_size=100, index=0)

In [ ]:
results = gdrive_pipeline.run_experiment_with_google_drive_saving('llama3', sample_size=100, index=0)


Running experiment with llama3 (with Google Drive saving)
Results will be saved to: /content/drive/MyDrive/LLMs/binary_classification/results/llama3
Loading data...
Data shape: (5348, 36)
Columns: 36
Preprocessing data...
Removed 0 rows with empty/short descriptions
Final data shape: (5348, 36)
Description length stats:
  Mean: 940.7
  Median: 626.0
  Max: 3456
Using sample of 100 records
Train set: 4278 samples
Test set: 1070 samples

[1/30] Classifying permission: Other : full network access


Classifying Other : full network access...:   0%|          | 0/1070 [00:00<?, ?it/s]

Loading llama3 (Llama 3 via HF + bnb 4-bit)...


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


llama3 loaded successfully!


Classifying Other : full network access...: 100%|██████████| 1070/1070 [42:28<00:00,  2.38s/it]


... (22 outputs left out)

  📄 Predictions saved: /content/drive/MyDrive/LLMs/binary_classification/results/llama3/predictions/Wi-Fi_connection_information___view_Wi-Fi_connections_20251129_153057.xlsx
  💾 Results saved: /content/drive/MyDrive/LLMs/binary_classification/results/llama3/json_results/Wi-Fi_connection_information___view_Wi-Fi_connections_20251129_153057.json
  📊 Visualization saved: /content/drive/MyDrive/LLMs/binary_classification/results/llama3/permission_plots/Wi-Fi_connection_information___view_Wi-Fi_connections_20251129_153057.png
✓ Completed Wi-Fi connection information : view Wi-Fi connections: Accuracy=0.350, F1=0.020

[13/30] Classifying permission: Location : precise location (GPS and network-based)


Classifying Location : precise location (G...: 100%|██████████| 1070/1070 [42:40<00:00,  2.39s/it]


  📄 Predictions saved: /content/drive/MyDrive/LLMs/binary_classification/results/llama3/predictions/Location___precise_location_(GPS_and_network-based)_20251129_161339.xlsx
  💾 Results saved: /content/drive/MyDrive/LLMs/binary_classification/results/llama3/json_results/Location___precise_location_(GPS_and_network-based)_20251129_161339.json
  📊 Visualization saved: /content/drive/MyDrive/LLMs/binary_classification/results/llama3/permission_plots/Location___precise_location_(GPS_and_network-based)_20251129_161339.png
✓ Completed Location : precise location (GPS and network-based): Accuracy=0.388, F1=0.000

[14/30] Classifying permission: Location : approximate location (network-based)


Classifying Location : approximate locatio...:  86%|████████▌ | 919/1070 [36:09<06:13,  2.47s/it]

In [ ]:
# results = gdrive_pipeline.run_experiment_with_google_drive_saving('mistral', sample_size=100, index=0)

## Custom Prompt Testing Interface

In [ ]:
def test_custom_prompt(prompt: str, model_name: str, max_new_tokens: int = 100,
                       temperature: float = 0.7, verbose: bool = True):
    """
    Send a custom prompt to any model for testing.

    Args:
        prompt: Custom prompt text to send to the model
        model_name: Name of the model ('llama2', 'mistral', 'flan-t5', 'phi-3')
        max_new_tokens: Maximum number of tokens to generate (default: 100)
        temperature: Sampling temperature (default: 0.7, use 0.0 for deterministic)
        verbose: Print detailed information (default: True)

    Returns:
        Generated response text
    """
    print(f"\n{'='*60}")
    print(f"Testing {model_name.upper()} with custom prompt")
    print(f"{'='*60}\n")

    if verbose:
        print(f"📝 Prompt: {prompt[:200]}{'...' if len(prompt) > 200 else ''}\n")

    try:
        # Initialize the appropriate classifier
        if model_name.lower() == 'llama2':
            classifier = Llama2Classifier(model_name, CONFIG)
        elif model_name.lower() == 'mistral':
            classifier = MistralClassifier(model_name, CONFIG)
        elif model_name.lower() == 'flan-t5':
            classifier = FlanT5Classifier(model_name, CONFIG)
        elif model_name.lower() == 'phi-3':
            classifier = Phi3Classifier(model_name, CONFIG)
        else:
            raise ValueError(f"Unknown model: {model_name}. Available: llama2, mistral, flan-t5, phi-3")

        # Load model if not already loaded
        if classifier.pipeline is None:
            classifier.load_model()

        # Generate response based on model type
        if model_name.lower() == 'flan-t5':
            # FLAN-T5 uses text2text-generation pipeline
            response = classifier.pipeline(
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0.0,
                temperature=temperature if temperature > 0.0 else 1.0,
                truncation=True
            )
        else:
            # Other models use text-generation pipeline
            # Check if model supports chat templates (Mistral, Llama2)
            use_chat_template = False
            if model_name.lower() in ['mistral', 'llama2']:
                if hasattr(classifier, 'tokenizer') and hasattr(classifier.tokenizer, 'apply_chat_template'):
                    use_chat_template = True

            if use_chat_template and model_name.lower() == 'mistral':
                # Use chat template for Mistral
                messages = [{"role": "user", "content": prompt}]
                formatted_prompt = classifier.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            else:
                formatted_prompt = prompt

            response = classifier.pipeline(
                formatted_prompt,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0.0,
                temperature=temperature,
                truncation=True,
                return_full_text=False  # Only return generated text
            )

        # Extract response text
        if isinstance(response, list):
            response_text = response[0].get('generated_text', '')
        else:
            response_text = response.get('generated_text', '')

        if verbose:
            print(f"🤖 Response:\n{response_text}\n")
            print(f"📊 Length: {len(response_text)} characters")

        return response_text

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        if verbose:
            traceback.print_exc()
        return None


# Example usage:
# test_custom_prompt(
#     prompt="Explain what machine learning is in one sentence.",
#     model_name="mistral",
#     max_new_tokens=50,
#     temperature=0.7
# )

print("✅ Custom prompt testing function ready!")
print("\nUsage example:")
print('test_custom_prompt(prompt="Your question here", model_name="mistral")')

In [ ]:
# app_description = 'We do not need access to your network.'

# desc2 = "Home Workouts provides daily workout routines for all your main muscle groups. In just a few minutes a day, you can <b>build muscles and keep fitness at home</b> without having to go to the gym. <b>No equipment</b> or coach needed, all exercises can be performed with just your body weight.\r\n\r\nThe app has workouts for your <b>abs, chest, legs, arms</b> and <b>butt</b> as well as full body workouts. All the workouts are designed by experts. None of them need equipment, so there's no need to go to the gym. Even though it just takes <b>a few minutes a day</b>, it can effectively tone your muscles and help you get six pack abs at home.\r\n\r\nThe warm-up and stretching routines are designed to make sure you exercise in a scientific way. With <b>animations and video guidance</b> for each exercise, you can make sure you use the right form during each exercise. \r\n\r\nStick with our home workouts, and you will notice a change in your body in just a few short weeks. 💪 💪 💪\r\n\r\n⭐ <b>Features</b> ⭐\r\n√ Warm-up and stretching routines\r\n√ Records training progress automatically \r\n√ The chart tracks your weight trends \r\n√ Customize your workout reminders \r\n√ Detailed video and animation guides \r\n√ Lose weight with a personal trainer \r\n√ Share with your friends on social media \r\n\r\nBodybuilding App\r\nLooking for a bodybuilding app? No satisfied bodybuilding app? Try our build muscle app! This build muscle app has effective muscle building workout, and all muscle building workout is designed by expert.\r\n\r\nStrength Training App\r\nIt's not only just a build muscle app, but also a strength training app. If you are still looking for muscle building workout, muscle building apps or strength training app, this muscle building apps is the best one you can find among the muscle building apps.\r\n\r\nFat Burning Workouts & HIIT Workouts\r\nThe best fat burning workouts & hiit workouts for better body shape. Burn calories with fat burning workouts, and combine with hiit workouts to get the best results.\r\n\r\nHome Workouts for Men\r\nWant effective home workouts for men? We provide different home workouts for men to workout at home. The home workout for men is proven to help you get six pack abs in a short time. You'll find the home workout for men that most suitable for you. Try our home workout for men now!\r\n\r\nMultiple Exercises\r\nPush ups, squats, sit ups, plank, crunch, wall sit, jumping jacks, punch, triceps dips, lunges...\r\n\r\nFitness Coach\r\nThe best fitness apps and workout apps. All sport and gym workout in this workout apps and fitness apps are designed by professional fitness coach. Sport and gym workout guide through the exercise, gym workout and sport, just like having a personal fitness coach in your pocket!"

# permission = 'Other : full network access'

In [ ]:
# test_custom_prompt(
#     prompt=f"""
# Task: Determine if an Android app requests the following permission based on its description.

# App Description: {app_description}

# Permission: {permission}

# Question: Does this app request the permission "{permission}"?

# Answer with only "YES" or "NO":""",
#     model_name="llama2"
#     )

## 8. Conclusion

This notebook provides a comprehensive pipeline for binary classification of Android app permissions using different Large Language Models (LLMs). The modular design allows for easy experimentation with different models and configurations, with advanced features for production use.

### 🚀 **Key Features:**

#### **Core Pipeline:**
- **Modular Design**: Easy to add new models or modify existing ones
- **4 LLM Support**: Llama 2, Mistral, FLAN-T5, and Phi-3
- **30 Target Permissions**: Comprehensive permission classification
- **Binary Classification**: Predicts if app requests specific permissions
- **Memory Efficient**: Uses quantization for large models
- **Scalable**: Can handle different sample sizes
- **Reproducible**: Fixed random seeds and structured results

#### **Advanced Features:**
- **🔄 Partial Result Saving**: Results saved after each permission completion
- **📁 Google Drive Integration**: Automatic mounting and organized storage
- **📂 Model-Specific Folders**: Separate folders for each model's results
- **📊 Rich Visualizations**: Individual and summary plots for each permission
- **⚡ Progress Tracking**: Real-time progress with [X/30] counter
- **🛡️ Error Resilience**: Graceful handling of interruptions and errors
- **💾 Structured Data**: JSON results with timestamps and metadata

### 📁 **Google Drive Organization:**
```
/content/drive/MyDrive/LLMs/binary_classification/results/
├── llama2/
│   ├── json_results/          # Individual & overall JSON results
│   ├── permission_plots/      # Individual permission visualizations
│   └── visualizations/        # Final summary plots
│   └── predictions/           # Predictions
├── mistral/
├── flan-t5/
└── phi-3/
```

### 🎯 **Use Cases:**
- **Research**: Compare LLM performance on permission classification
- **Production**: Deploy permission prediction systems
- **Analysis**: Understand which permissions are easier/harder to predict
- **Benchmarking**: Evaluate different models on the same dataset

### 🚀 **Quick Start:**

#### **Basic Usage:**
```python
# Original pipeline
results = classification_pipeline.run_experiment('flan-t5', sample_size=100)

# Google Drive pipeline (recommended)
results = gdrive_pipeline.run_experiment_with_google_drive_saving('flan-t5', sample_size=100)
```

#### **Advanced Usage:**
```python
# Check saved results
model_path = gdrive_pipeline.get_model_path('flan-t5')
print('Files:', os.listdir(f'{model_path}/json_results'))

# Load previous results
with open(f'{model_path}/json_results/flan-t5_overall_results.json', 'r') as f:
    saved_results = json.load(f)
```

### 📊 **Evaluation Metrics:**
- **Accuracy**: Overall correctness
- **Precision**: True positives / (True positives + False positives)
- **Recall**: True positives / (True positives + False negatives)
- **F1-Score**: Harmonic mean of precision and recall

### 🔧 **Model Recommendations:**

| Model | Speed | Memory | Accuracy | Best For |
|-------|-------|--------|----------|----------|
| FLAN-T5 | ⚡⚡⚡ | 💾💾 | ⭐⭐⭐ | Quick testing, baseline |
| Phi-3 | ⚡⚡ | 💾💾💾 | ⭐⭐⭐⭐ | Balanced performance |
| Mistral | ⚡ | 💾💾💾💾 | ⭐⭐⭐⭐ | Good accuracy/speed |
| Llama 2 | ⚡ | 💾💾💾💾💾 | ⭐⭐⭐⭐⭐ | Best accuracy |

### 🛠️ **Troubleshooting:**
- **Out of Memory**: Use smaller models or reduce sample size
- **Slow Performance**: Start with FLAN-T5, use smaller samples
- **Google Drive Issues**: Check mounting permissions
- **Interrupted Execution**: Results are automatically saved after each permission

### 📈 **Next Steps:**
1. **Start Small**: Test with sample sizes of 10-100
2. **Scale Up**: Increase sample size as you gain confidence
3. **Compare Models**: Run all models on the same data
4. **Analyze Results**: Use visualizations to understand performance patterns
5. **Customize**: Modify prompts or add new models as needed

### 💡 **Pro Tips:**
- Use T4 GPU in Google Colab for best performance
- Save results frequently to avoid losing work
- Start with FLAN-T5 for quick testing
- Use the Google Drive pipeline for production
- Check the model-specific folders for organized results

### 🎉 **Ready to Use:**
The pipeline is production-ready with comprehensive error handling, automatic saving, and rich visualizations. Choose the appropriate pipeline based on your needs and start experimenting with Android app permission classification!
